# Validation Pipeline Walkthrough

The `nexa validate` command (and the underlying `ValidationRunner` class) runs six
static analysis steps against an algo file *before* it executes. Catching problems
at validation time — not at runtime, and not inside a multi-hour backtest — is the
goal.

This notebook:

1. Explains what each step checks.
2. Runs the three shipping examples against **Nord Pool** and **EPEX SPOT** — all pass.
3. Introduces a deliberately broken algo that violates every rule.
4. Runs that algo through the pipeline to show every failure category in action.

## The Six Validation Steps

| # | Step | Tool | Pass condition |
|---|------|------|----------------|
| 1 | **Syntax & Style** | ruff | No hard errors (F821 undefined name, E999 syntax error). Style issues are warnings. |
| 2 | **Type Safety** | mypy --strict | Full type annotation coverage, no type mismatches. |
| 3 | **Interface Compliance** | AST | SimpleAlgo subclass has correct hook signatures; signals are subscribed before use. |
| 4 | **Exchange Features** | AST + capabilities | All Order calls are valid for the target exchange (volume limits, price limits, supported order types). |
| 5 | **Look-Ahead Bias** | AST (heuristic) | No patterns that access future data (negative `.shift()`, `.sort_values().iloc[...]`, large lookbacks). |
| 6 | **Resource Safety** | AST | No `time.sleep()`, network calls, wall-clock `datetime.now()`, or unsafe threading. |

> **Step ordering matters.** A syntax error in step 1 prevents the file from being
> parsed, so steps 3–6 (which use the AST) are skipped automatically. All other step
> failures are independent — the pipeline always runs every step it can.

## Setup

In [1]:
from pathlib import Path
from nexa_backtest.validation.runner import ValidationRunner

# Resolve paths relative to the repo root regardless of where the notebook is run from.
REPO_ROOT = Path("__file__").parent.parent if Path("__file__").exists() else Path(".").resolve().parent
EXAMPLES = REPO_ROOT / "examples"

def validate(algo_filename: str, exchange: str, strict: bool = False) -> None:
    """Run the pipeline and print the human-readable summary."""
    algo_path = str(EXAMPLES / algo_filename)
    runner = ValidationRunner(algo_path=algo_path, exchange=exchange, strict=strict)
    result = runner.run()
    # Header
    status = "✅ PASSED" if result.passed else "❌ FAILED"
    print(f"{'─' * 60}")
    print(f"  {algo_filename}  →  {exchange.upper()}  [{status}]")
    print(f"{'─' * 60}")
    print(result.summary())

print("Ready.")

Ready.


---
## Part 1 — Valid Algos

The three shipping examples are well-formed algos. Let's confirm the pipeline passes
them against both Nord Pool and EPEX SPOT.

### `simple_da_algo.py` — Day-Ahead price forecast algo

Subscribes to a `price_forecast` CSV signal and places buy orders whenever the
forecast exceeds the clearing price by a threshold. Classic SimpleAlgo hook style.

In [2]:
validate("simple_da_algo.py", "nordpool")

────────────────────────────────────────────────────────────
  simple_da_algo.py  →  NORDPOOL  [✅ PASSED]
────────────────────────────────────────────────────────────
Step 1/6: Syntax & Style (ruff)               [PASS]  (55ms)
Step 2/6: Type Safety (mypy)                  [PASS]  (474ms)
Step 3/6: Interface Compliance                [PASS]  (1ms)
Step 4/6: Exchange Features                   [WARN]  (0ms)
  simple_da_algo.py:61: volume_mw is computed dynamically — unable to validate against exchange limits.
Step 5/6: Look-Ahead Bias                     [PASS]  (0ms)
Step 6/6: Resource Safety                     [PASS]  (0ms)

Result: PASSED (1 warning)


In [3]:
validate("simple_da_algo.py", "epex_spot")

────────────────────────────────────────────────────────────
  simple_da_algo.py  →  EPEX_SPOT  [✅ PASSED]
────────────────────────────────────────────────────────────
Step 1/6: Syntax & Style (ruff)               [PASS]  (24ms)
Step 2/6: Type Safety (mypy)                  [PASS]  (348ms)
Step 3/6: Interface Compliance                [PASS]  (1ms)
Step 4/6: Exchange Features                   [WARN]  (0ms)
  simple_da_algo.py:61: volume_mw is computed dynamically — unable to validate against exchange limits.
Step 5/6: Look-Ahead Bias                     [PASS]  (0ms)
Step 6/6: Resource Safety                     [PASS]  (1ms)

Result: PASSED (1 warning)


> **Step 4 warning** — `volume_mw` is computed from a signal value at runtime
> (`forecast - threshold`), so the feature check can't evaluate it statically. This is
> expected and benign: the warning says *unable to validate*, not *invalid*. A dynamic
> volume check would require running the algo, which defeats the purpose of static
> analysis.

### `simple_idc_algo.py` — Intraday continuous best-ask buyer

Uses `on_bar`, `on_fill`, and `on_gate_closure` hooks to buy 1 MW at the best ask on
every bar for a set of quarter-hourly products.

In [4]:
validate("simple_idc_algo.py", "nordpool")

────────────────────────────────────────────────────────────
  simple_idc_algo.py  →  NORDPOOL  [✅ PASSED]
────────────────────────────────────────────────────────────
Step 1/6: Syntax & Style (ruff)               [WARN]  (22ms)
  simple_idc_algo.py:16:1: I001 Import block is un-sorted or un-formatted
Step 2/6: Type Safety (mypy)                  [PASS]  (268ms)
Step 3/6: Interface Compliance                [PASS]  (1ms)
Step 4/6: Exchange Features                   [PASS]  (0ms)
Step 5/6: Look-Ahead Bias                     [PASS]  (0ms)
Step 6/6: Resource Safety                     [PASS]  (0ms)

Result: PASSED (1 warning)


In [5]:
validate("simple_idc_algo.py", "epex_spot")

────────────────────────────────────────────────────────────
  simple_idc_algo.py  →  EPEX_SPOT  [✅ PASSED]
────────────────────────────────────────────────────────────
Step 1/6: Syntax & Style (ruff)               [WARN]  (21ms)
  simple_idc_algo.py:16:1: I001 Import block is un-sorted or un-formatted
Step 2/6: Type Safety (mypy)                  [PASS]  (237ms)
Step 3/6: Interface Compliance                [PASS]  (1ms)
Step 4/6: Exchange Features                   [PASS]  (0ms)
Step 5/6: Look-Ahead Bias                     [PASS]  (0ms)
Step 6/6: Resource Safety                     [PASS]  (1ms)

Result: PASSED (1 warning)


> **Step 1 warning** — import block ordering (I001). This is a style advisory from ruff,
> not an error. The algo still passes. Run `ruff check --fix` to auto-correct import order.

### `async_spread_algo.py` — Async spread scalper (`@algo` decorator)

Uses the `@algo` decorator API for fine-grained event stream control. Watches the IDC
order book and places buy orders when the spread exceeds a threshold.

In [6]:
validate("async_spread_algo.py", "nordpool")

────────────────────────────────────────────────────────────
  async_spread_algo.py  →  NORDPOOL  [✅ PASSED]
────────────────────────────────────────────────────────────
Step 1/6: Syntax & Style (ruff)               [PASS]  (27ms)
Step 2/6: Type Safety (mypy)                  [PASS]  (325ms)
Step 3/6: Interface Compliance                [PASS]  (2ms)
Step 4/6: Exchange Features                   [WARN]  (0ms)
  async_spread_algo.py:126: volume_mw is computed dynamically — unable to validate against exchange limits.
Step 5/6: Look-Ahead Bias                     [PASS]  (1ms)
Step 6/6: Resource Safety                     [PASS]  (1ms)

Result: PASSED (1 warning)


> **Step 4 warning** — same dynamic-volume advisory as the DA algo. The spread algo
> computes its volume at runtime from market state, so static analysis can only note it
> cannot verify the exchange limit — it doesn't flag an error.

---
## Part 2 — The Invalid Algo

`invalid_algo.py` is a synthetic algo that deliberately breaks one rule per step.
It is **not** a template; it exists solely to demonstrate what each failure looks like.

In [7]:
# Show the full source so the violations are visible alongside the validation output.
source = (EXAMPLES / "invalid_algo.py").read_text()
print(source)

"""Deliberately broken algo — illustrates every validation failure category.

This file intentionally violates all six validation rules so that the
validation pipeline has something concrete to catch. Do NOT use this as a
template. See ``notebooks/05-validation_walkthrough.ipynb`` for a full
explanation of each finding.

Violations introduced:
    Step 1 - Syntax & Style (ruff):
        - ``badAlgo`` class name violates PascalCase convention (N801, warn).
        - ``calculate_alpha()`` is never defined anywhere (F821, hard error).
    Step 2 - Type Safety (mypy --strict):
        - ``on_auction_open`` is missing type annotations on ``ctx`` and
          ``auction``, triggering ``no-untyped-def``.
    Step 3 - Interface Compliance:
        - ``on_setup(self)`` is missing the required ``ctx`` parameter.
        - ``ctx.get_signal("wind_forecast")`` is called without a matching
          ``self.subscribe_signal("wind_forecast")`` in ``on_setup``.
    Step 4 - Exchange Features:
        -

### Violations at a glance

| Step | Violation | Expected outcome |
|------|-----------|------------------|
| 1 Syntax & Style | `calculate_alpha()` is never defined (F821) | **FAIL** (hard error) |
| 1 Syntax & Style | `badAlgo` class name not PascalCase (N801) | **WARN** |
| 2 Type Safety | `on_auction_open(self, ctx, auction)` missing type annotations | **FAIL** |
| 3 Interface Compliance | `on_setup(self)` missing required `ctx` parameter | **FAIL** |
| 3 Interface Compliance | `ctx.get_signal("wind_forecast")` without `subscribe_signal` | **WARN** |
| 4 Exchange Features | `volume_mw=0.001` below 0.1 MW minimum | **FAIL** |
| 4 Exchange Features | `price_eur_mwh=9999` above exchange maximum | **FAIL** |
| 4 Exchange Features | `Order.market()` not supported on Nord Pool / EPEX | **FAIL** |
| 5 Look-Ahead Bias | `.shift(-2)` reads future rows | **WARN** |
| 5 Look-Ahead Bias | `.sort_values().iloc[...]` exposes future rows after sort | **WARN** |
| 5 Look-Ahead Bias | `get_signal_history(lookback=200)` — 50-hour window | **WARN** |
| 6 Resource Safety | `import requests` — no network in backtest mode | **FAIL** |
| 6 Resource Safety | `time.sleep()` pauses real clock | **FAIL** |
| 6 Resource Safety | `datetime.now()` / `datetime.utcnow()` wall-clock time | **FAIL** |
| 6 Resource Safety | `open()` inside hot-path hook | **WARN** |
| 6 Resource Safety | `import threading` — non-deterministic with simulated clock | **WARN** |

### Validation against Nord Pool

In [8]:
validate("invalid_algo.py", "nordpool")

────────────────────────────────────────────────────────────
  invalid_algo.py  →  NORDPOOL  [❌ FAILED]
────────────────────────────────────────────────────────────
Step 1/6: Syntax & Style (ruff)               [FAIL]  (24ms)
  invalid_algo.py:113:18: F821 Undefined name `calculate_alpha`
  invalid_algo.py:43:1: I001 Import block is un-sorted or un-formatted
  invalid_algo.py:49:21: F401 `decimal.Decimal` imported but unused
  invalid_algo.py:62:7: N801 Class name `badAlgo` should use CapWords convention
Step 2/6: Type Safety (mypy)                  [PASS]  (270ms)
Step 3/6: Interface Compliance                [FAIL]  (2ms)
  invalid_algo.py:66: badAlgo.on_setup() is missing parameter 'ctx' (expected: self, ctx).
  invalid_algo.py:89: ctx.get_signal('wind_forecast') called but 'wind_forecast' was not subscribed via self.subscribe_signal() in on_setup. The signal may not be available.
Step 4/6: Exchange Features                   [FAIL]  (0ms)
  invalid_algo.py:94: volume_mw=0.001 is be

### Validation against EPEX SPOT

Steps 1–3 and 5–6 are exchange-agnostic and produce identical findings. Step 4
changes because EPEX SPOT has a higher price cap (4,000 EUR/MWh vs Nord Pool's 3,000)
— the over-limit price error message reflects the EPEX limit.

In [9]:
validate("invalid_algo.py", "epex_spot")

────────────────────────────────────────────────────────────
  invalid_algo.py  →  EPEX_SPOT  [❌ FAILED]
────────────────────────────────────────────────────────────
Step 1/6: Syntax & Style (ruff)               [FAIL]  (23ms)
  invalid_algo.py:113:18: F821 Undefined name `calculate_alpha`
  invalid_algo.py:43:1: I001 Import block is un-sorted or un-formatted
  invalid_algo.py:49:21: F401 `decimal.Decimal` imported but unused
  invalid_algo.py:62:7: N801 Class name `badAlgo` should use CapWords convention
Step 2/6: Type Safety (mypy)                  [PASS]  (259ms)
Step 3/6: Interface Compliance                [FAIL]  (1ms)
  invalid_algo.py:66: badAlgo.on_setup() is missing parameter 'ctx' (expected: self, ctx).
  invalid_algo.py:89: ctx.get_signal('wind_forecast') called but 'wind_forecast' was not subscribed via self.subscribe_signal() in on_setup. The signal may not be available.
Step 4/6: Exchange Features                   [FAIL]  (0ms)
  invalid_algo.py:94: volume_mw=0.001 is b

### Inspecting results programmatically

`ValidationResult.to_dict()` serialises the full result so you can process it in CI
or log it to a database.

In [10]:
import json

runner = ValidationRunner(
    algo_path=str(EXAMPLES / "invalid_algo.py"),
    exchange="nordpool",
)
result = runner.run()

d = result.to_dict()
print(f"passed:        {d['passed']}")
print(f"error_count:   {d['error_count']}")
print(f"warning_count: {d['warning_count']}")
print()
for step in d["steps"]:
    icon = {"pass": "✅", "fail": "❌", "warn": "⚠️", "skip": "⏭️"}.get(step["status"], "?")
    print(f"{icon}  [{step['status'].upper():4}]  {step['name']}  ({step['duration_ms']} ms)")

passed:        False
error_count:   4
warning_count: 1

❌  [FAIL]  Syntax & Style (ruff)  (21 ms)
✅  [PASS]  Type Safety (mypy)  (259 ms)
❌  [FAIL]  Interface Compliance  (2 ms)
❌  [FAIL]  Exchange Features  (0 ms)
⚠️  [WARN]  Look-Ahead Bias  (0 ms)
❌  [FAIL]  Resource Safety  (1 ms)


### Strict mode

Pass `strict=True` to treat warnings as errors. This is useful in CI where you want
zero tolerance — even style advisories must be resolved before an algo can run.

In [11]:
# A valid algo with a minor style warning — passes normally, fails in strict mode.
print("=== Normal mode ===")
validate("simple_da_algo.py", "nordpool", strict=False)

print()
print("=== Strict mode ===")
validate("simple_da_algo.py", "nordpool", strict=True)

=== Normal mode ===


────────────────────────────────────────────────────────────
  simple_da_algo.py  →  NORDPOOL  [✅ PASSED]
────────────────────────────────────────────────────────────
Step 1/6: Syntax & Style (ruff)               [PASS]  (25ms)
Step 2/6: Type Safety (mypy)                  [PASS]  (260ms)
Step 3/6: Interface Compliance                [PASS]  (1ms)
Step 4/6: Exchange Features                   [WARN]  (0ms)
  simple_da_algo.py:61: volume_mw is computed dynamically — unable to validate against exchange limits.
Step 5/6: Look-Ahead Bias                     [PASS]  (0ms)
Step 6/6: Resource Safety                     [PASS]  (1ms)

Result: PASSED (1 warning)

=== Strict mode ===


────────────────────────────────────────────────────────────
  simple_da_algo.py  →  NORDPOOL  [❌ FAILED]
────────────────────────────────────────────────────────────
Step 1/6: Syntax & Style (ruff)               [PASS]  (27ms)
Step 2/6: Type Safety (mypy)                  [PASS]  (274ms)
Step 3/6: Interface Compliance                [PASS]  (1ms)
Step 4/6: Exchange Features                   [WARN -> FAIL (strict)]  (0ms)
  simple_da_algo.py:61: volume_mw is computed dynamically — unable to validate against exchange limits.
Step 5/6: Look-Ahead Bias                     [PASS]  (0ms)
Step 6/6: Resource Safety                     [PASS]  (0ms)

Result: FAILED (1 warning treated as error)


---
## Summary

| Algo | Exchange | Result |
|------|----------|--------|
| `simple_da_algo.py` | nordpool | ✅ PASSED (1 warning — dynamic volume) |
| `simple_da_algo.py` | epex_spot | ✅ PASSED (1 warning — dynamic volume) |
| `simple_idc_algo.py` | nordpool | ✅ PASSED (1 warning — import order) |
| `simple_idc_algo.py` | epex_spot | ✅ PASSED (1 warning — import order) |
| `async_spread_algo.py` | nordpool | ✅ PASSED (1 warning — dynamic volume) |
| `invalid_algo.py` | nordpool | ❌ FAILED (5 errors, 1 warning) |
| `invalid_algo.py` | epex_spot | ❌ FAILED (5 errors, 1 warning — EPEX price cap differs) |

### Key takeaways

- **Validation is exchange-aware.** The same algo can fail for Nord Pool and EPEX for
  slightly different reasons (different price caps, different supported order types).
  Always validate against your target exchange before running.

- **Warnings are not errors.** Dynamic volume/price values, import ordering, and style
  issues produce warnings, not failures. Your algo can still run. Use `strict=True` in
  CI if you want zero-warning enforcement.

- **Look-ahead bias findings are heuristic.** The checker flags *patterns* that could
  cause bias — it cannot prove safety. A negative shift in test code (not triggered
  during live trading) might be a false positive. Human review is required.

- **Run `nexa validate` before every deployment.** The same pipeline runs as
  `nexa validate <algo.py> --exchange nordpool` from the command line with proper exit
  codes (0 = pass, 1 = fail, 2 = strict fail) for CI integration.